email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass
@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage("Successfully authenticated", tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage("Authentication failed", tool_call_id=runtime.tool_call_id)]
        })

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie,
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only allow inbox/send tools once authenticated; otherwise only allow authenticate"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt_func(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")
    return authenticated_prompt if authenticated else unauthenticated_prompt


In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    checkpointer=InMemorySaver(),
    middleware=[
        dynamic_tool_call,
        dynamic_prompt_func,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            }
        ),
    ],
)


In [17]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)


I can help with inbox tasks, but I can’t verify or store passwords here. If you’d like, I can check your inbox for recent emails and proceed from there. What would you like to do next?

Options:
- Check latest emails and summarize
- Read or reply to a specific message (tell me which one)
- Draft or send a new email (provide recipient, subject, and body)
- Search for messages from a specific sender or with a keyword

If you want me to proceed with checking the inbox now, just say “check inbox.”


In [18]:
response = agent.invoke(
    {"messages": [HumanMessage(content="check inbox")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s the latest message I found:

From: Jane (jane@example.com)
Message: Hi Julie, I'm going to be in town next week and was wondering if we could grab a coffee? - best, Jane (jane@example.com)

Summary: Jane will be in town next week and would like to meet for coffee. She’s asking to set up a time.

Would you like me to reply? I can draft a response for you. Here are a couple of options you can choose from, or I can tailor one:

Option A — casual and open
Subject: Re: Coffee next week
Hi Jane,
That sounds great! I’d love to catch up. I’m free Tuesday or Thursday afternoon next week. Do either of those work for you? If you have a preferred place, let me know.
Best,
Julie

Option B — brief and direct
Subject: Re: Coffee next week
Hi Jane, I’d love that. I’m free Tuesday and Thursday afternoon next week—tell me what works and we’ll pick a time.
Julie

Option C — formal
Subject: Re: Coffee next week
Hi Jane,
Thank you for reaching out. I’d be happy to meet for coffee while you’re in tow

In [19]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Option A")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [20]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,
That sounds great! I’d love to catch up. I’m free Tuesday or Thursday afternoon next week. Do either of those work for you? If you have a preferred place, let me know.
Best,
Julie


In [21]:
from langgraph.types import Command

response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)

print(response['messages'][-1].content)


Email sent successfully to jane@example.com with subject "Re: Coffee next week."

Summary of what I sent:
- Hi Jane,
  That sounds great! I’d love to catch up. I’m free Tuesday or Thursday afternoon next week. Do either of those work for you? If you have a preferred place, let me know.
  Best,
  Julie

Would you like me to:
- Check for Jane’s reply when it comes in and draft a follow-up if needed?
- Propose a couple of tentative times/locations based on her response?
- Do anything else with this thread (e.g., draft a calendar invite or a backup plan)?


In [22]:
from pprint import pprint
pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='4070d5be-cbe3-4cba-b236-61d09ad7a8a7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 151, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EB10cR9cxRr3lp5WMsVfUPvmbMJXk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe76e-4c10-7261-811d-f9e0f21344bc-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_swdISzsBxXmlgyNAQNgCE9ND', 'type': 'tool_call